In [1]:
from datasets import load_dataset
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import BartForConditionalGeneration, BartTokenizer, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

C:\Users\Akshay\anaconda3\envs\tf\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Title Extraction

## Datasets

In [2]:
# from datasets import load_dataset
# dataset = load_dataset("sentence-transformers/reddit-title-body")

In [3]:
# dataset

In [4]:
# train_df_1 = pd.DataFrame()
# train_df_1["title"] = dataset["train"]["title"][:1000000]
# train_df_1["body"] = dataset["train"]["body"][:1000000]
# train_df_1.to_csv("titleExtractionDataset.csv")

In [2]:
dataset_1 = pd.read_csv("../Data/CleanedDatasets/titleExtractionDataset.csv")

In [3]:
dataset_1.drop("Unnamed: 0",inplace = True,axis = 1)
dataset_1.columns = ["summary","text"]

In [2]:
dataset_2 = load_dataset("Ateeqq/news-title-generator")

In [8]:
dataset_2

DatasetDict({
    train: Dataset({
        features: ['summary', 'text'],
        num_rows: 98400
    })
})

In [9]:
# dataset_2["train"]["summary"][:10]

In [10]:
# dataset_2["train"]["text"][:10]

In [11]:
type(list(dataset_1["text"]))

list

In [2]:
dataset_3 = load_dataset("dreamproit/bill_summary_us")

In [1]:
# dataset_3["train"]["text"]

In [2]:
# dataset_3["train"]["title"]

## Train Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(dataset_2["train"]["text"],dataset_2["train"]["summary"], test_size=0.2, random_state=42)
# X_train, X_test, y_train, y_test = train_test_split(dataset_1["text"],dataset_1["summary"], test_size=0.2, random_state=42)
X_test, X_valid, y_test, y_valid = train_test_split(X_test,y_test, test_size=0.5, random_state=42)

## Loading the Model

In [5]:
import torch
torch.cuda.empty_cache()
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(torch.cuda.current_device()))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

CUDA available: True
CUDA version: 12.1
Device count: 1
Current device: 0
Device name: NVIDIA GeForce RTX 4090
cuda


In [5]:
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large').to(device)
model = model.to(device)

In [6]:
model_name = '../Saved_Models/TitleExtraction/fine-tuned-bert-sentiment_01-07-2024_1'
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)
model = model.to(device)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [6]:
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, input_texts, output_texts, tokenizer, max_length):
        self.input_texts = input_texts
        self.output_texts = output_texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.input_texts)

    def __getitem__(self, idx):
        input_text = self.input_texts[idx]
        output_text = self.output_texts[idx]

        input_tokenized = self.tokenizer(input_text, max_length=self.max_length, truncation=True, padding='max_length', return_tensors='pt')
        output_tokenized = self.tokenizer(output_text, max_length=self.max_length, truncation=True, padding='max_length', return_tensors='pt')

        input_ids = input_tokenized['input_ids'].squeeze(0)
        attention_mask = input_tokenized['attention_mask'].squeeze(0)
        labels = output_tokenized['input_ids'].squeeze(0)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

## Training the model

In [9]:
train_dataset = TextDataset(X_train, y_train, tokenizer, max_length=128)
valdiation_dataset = TextDataset(X_valid, y_valid, tokenizer, max_length=128)

In [10]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,  # Increase the number of epochs
    per_device_train_batch_size=90,  # Adjust batch size as per your GPU memory
    per_device_eval_batch_size=90,
    gradient_accumulation_steps=2,  # Gradient accumulation to simulate larger batch size
    fp16=True,  # Enable mixed precision training
    learning_rate=3e-5,  # Adjust learning rate
    logging_dir='./logs',
    logging_steps=100,
    evaluation_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    weight_decay=0.01,  # Regularization
    warmup_steps=500,  # Learning rate warmup
)

In [11]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset = valdiation_dataset,
    data_collator=data_collator,
)

# Train the model
trainer.train()

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [20]:
model.save_pretrained('./Saved_Models/TitleExtraction/fine-tuned-bert-sentiment_{}'.format("30-06-2024"))
tokenizer.save_pretrained('./Saved_Models/TitleExtraction/fine-tuned-bert-sentiment_{}'.format("30-06-2024"))
trainer.save_state()

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}


## Testing the model

In [7]:
X_test = list(X_test)
y_test = list(y_test)

In [8]:
for i in range(0,len(X_test[:20])):
    # print(X_test[i])
    input_text = X_test[i]
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(device)
    output = model.generate(
        input_ids, 
        max_length=128, 
        num_beams=5,
        early_stopping=True, 
        no_repeat_ngram_size=2,  # Prevent repeating n-grams
        num_return_sequences=1,  # Number of sequences to return
        temperature=0.7,  # Sampling temperature
        top_k=50,  # Top-K sampling
        top_p=0.9  # Top-p (nucleus) sampling
    )
    response = tokenizer.decode(output[0], skip_special_tokens=True)

    print("OG:",y_test[i])
    print("Answer:", response)
    print()

OG: Bradman's 100th first-class hundred came against an Indian side
Answer: Bradmanman hitmanstst hisst 100 100st'sst centurystthst

OG: K'taka man held for molesting UK citizen at Mumbai airport
Answer: K arrested arrested mol molestingesting foresting in in at in airport at airport airport

OG: FIFA paying Maradona â¹9L per day to attend Russia WC: Report
Answer: F pays pays paying pays � ��� Â¹����9� to�,�

OG: Wedding crashers apologise, leave $1 as gift for newlyweds
Answer: W wedding wedding cr cr couple couple a couple for for a a for polar polar for

OG: Trump wants Ivanka, Kushner out of White House: Report
Answer: Trump seeks seeks to help help to to Kelly Kelly to Kushner Kushner out Kushner

OG: Paris climate deal could make the world $19 trillion richer
Answer: Invest investment in to to could could to by $ by by:: by19:

OG: 4 youths held for collecting funds with fake DU letterhead
Answer: 4 youths youths&& for& funds for for head head letter head with head

OG: Vinod 

803899    This is such a non-victory it's silly, but I'm...
624862    I hear a lot about Pawsitive Dog and Cooperati...
666323    All they did was supply the fertilizer; the we...
742056    I've never played Doom 3.  Is it worth buying?...
896804    I met this guy right before Valentine's Day. W...
                                ...                        
146165    Remember how you used to play those old 8-bit ...
927253    My girlfriend goes to college about an half ho...
392080    Sex is a "zesty enterprise:"  \nGender more co...
669591    The wage gap is often discussed, but statistic...
683726    \n\nBooks!\n\n\n\nHello again, trusty 'ole Rog...
Name: text, Length: 100000, dtype: object

Current Date and Time: 2024_07_01


In [5]:
now

datetime.datetime(2024, 7, 1, 11, 8, 49, 271803)